# F1 Telemetry Data Pipeline (Colab)

Downloads race telemetry for 2018-2024 using FastF1, produces 3 parquet files:
- `telemetry.parquet` — X, Y, speed, throttle, brake, t per driver per lap
- `circuits.parquet` — track outline polyline per circuit
- `track_status.parquet` — safety car, VSC, red flag events

**Run all cells, then download the files from `/content/output/`**

To keep it running after closing your laptop:
1. Click Runtime → Change runtime type → keep as CPU (no GPU needed)
2. Run all cells
3. Close your laptop — Colab keeps running for ~90 min of idle time
4. Come back and download the files

In [ ]:
# Install FastF1
!pip install -q fastf1 pyarrow

In [ ]:
import sys
import warnings
import time
import fastf1
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import clear_output

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Cache in Colab's ephemeral storage (speeds up retries within same session)
CACHE_DIR = Path("/content/cache/fastf1")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
fastf1.Cache.enable_cache(str(CACHE_DIR))

OUTPUT_DIR = Path("/content/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEASONS = [2018, 2019, 2020, 2021, 2022, 2023, 2024]
TELEMETRY_POINTS_PER_LAP = 100

print(f"FastF1 v{fastf1.__version__}")
print(f"Cache: {CACHE_DIR}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
def downsample_telemetry(tel_df, n_points=TELEMETRY_POINTS_PER_LAP):
    if tel_df is None or tel_df.empty:
        return None
    tel_df = tel_df.copy()
    time_col = "SessionTime" if "SessionTime" in tel_df.columns else "Time"
    tel_df["t"] = tel_df[time_col].dt.total_seconds()
    cols = ["X", "Y", "Distance", "Speed", "Throttle", "Brake", "t"]
    tel_df = tel_df.sort_values("Distance").reset_index(drop=True)
    if len(tel_df) <= n_points:
        return tel_df[cols].copy()
    indices = np.linspace(0, len(tel_df) - 1, n_points, dtype=int)
    return tel_df.iloc[indices][cols].reset_index(drop=True)


def session_driver_mapping(session):
    mapping = {}
    try:
        res = session.results
        if res is None or res.empty:
            return mapping
        for _, row in res.iterrows():
            abbr = str(row.get("Abbreviation", "") or "")
            did = str(row.get("DriverId", "") or "")
            if abbr and did:
                mapping[abbr] = did
            # Fallback: build from first/last name
            if abbr and not did:
                first = str(row.get("FirstName", "") or "").lower()
                last = str(row.get("LastName", "") or "").lower()
                if first and last:
                    mapping[abbr] = f"{first}_{last}"
    except Exception:
        pass
    return mapping


def extract_telemetry(session, race_id, driver_mapping):
    rows = []
    try:
        laps = session.laps
    except Exception:
        return rows
    if laps is None or laps.empty:
        return rows

    for driver in laps["Driver"].unique():
        driver_laps = laps.pick_drivers(driver)
        if driver_laps.empty:
            continue
        driver_id = driver_mapping.get(driver, driver.lower())
        for _, lap in driver_laps.iterrows():
            lap_num = int(lap["LapNumber"])
            try:
                tel = lap.get_telemetry()
                if tel is None or tel.empty:
                    continue
                sampled = downsample_telemetry(tel)
                if sampled is None or sampled.empty:
                    continue
                for i, row in sampled.iterrows():
                    rows.append({
                        "race_id": race_id,
                        "driver": driver_id,
                        "lap_number": lap_num,
                        "sample_index": i,
                        "x": round(float(row["X"]), 1),
                        "y": round(float(row["Y"]), 1),
                        "distance": round(float(row["Distance"]), 1),
                        "speed": round(float(row["Speed"]), 1),
                        "throttle": round(float(row["Throttle"]), 1) if pd.notna(row["Throttle"]) else 0.0,
                        "brake": bool(row["Brake"]) if pd.notna(row["Brake"]) else False,
                        "t_raw": round(float(row["t"]), 3),
                    })
            except Exception:
                continue
    return rows


def extract_track_status(session, race_id):
    rows = []
    try:
        status = session.track_status
        if status is None or status.empty:
            return rows
        for _, event in status.iterrows():
            sc = str(event.get("Status", ""))
            tv = event.get("Time")
            ts = None
            if not pd.isna(tv):
                try: ts = round(tv.total_seconds(), 2)
                except: ts = None
            label = {"1":"Green","2":"Yellow","4":"SafetyCar","5":"Red","6":"VSC","7":"VSCEnding"}.get(sc, f"Unknown({sc})")
            rows.append({"race_id": race_id, "time_seconds": ts, "status_code": sc, "status": label})
    except Exception:
        pass
    return rows


def extract_circuit_geometry(session, race_id, circuit_name):
    try:
        fastest = session.laps.pick_fastest()
        if fastest is None: return None
        tel = fastest.get_telemetry()
        if tel is None or tel.empty: return None
        try:
            ci = session.get_circuit_info()
            rotation = float(ci.rotation) if ci else 0.0
        except: rotation = 0.0
        tel = tel.sort_values("Distance").reset_index(drop=True)
        n_points = min(200, len(tel))
        indices = np.linspace(0, len(tel) - 1, n_points, dtype=int)
        sampled = tel.iloc[indices]
        rows = []
        for _, row in sampled.iterrows():
            rows.append({
                "race_id": race_id, "circuit_name": circuit_name,
                "rotation": rotation, "point_index": len(rows),
                "x": round(float(row["X"]), 1), "y": round(float(row["Y"]), 1),
                "distance": round(float(row["Distance"]), 1),
            })
        return rows
    except: return None

print("Helper functions ready.")

In [ ]:
# Build the list of all races to process
all_events = []
for season in SEASONS:
    schedule = fastf1.get_event_schedule(season, include_testing=False)
    races = schedule[schedule["EventFormat"] != "testing"]
    for _, event in races.iterrows():
        all_events.append((season, int(event["RoundNumber"]), event["EventName"]))

total_races = len(all_events)
print(f"Total races to fetch: {total_races}")
print(f"Seasons: {SEASONS[0]}-{SEASONS[-1]}")
for s in SEASONS:
    count = sum(1 for e in all_events if e[0] == s)
    print(f"  {s}: {count} races")

In [ ]:
# ===== MAIN FETCH LOOP =====
# This cell takes ~30-60 min on Colab. Progress is printed live.
# Safe to close your laptop — Colab keeps running.

all_telemetry = []
all_track_status = []
all_circuits = {}
errors = []
start_time = time.time()

for i, (season, round_num, event_name) in enumerate(all_events):
    race_id = f"{season}_{round_num}"

    elapsed = time.time() - start_time
    if i > 0:
        eta = elapsed / i * (total_races - i)
        eta_str = f"ETA: {int(eta//60)}m{int(eta%60)}s"
    else:
        eta_str = "ETA: calculating..."

    pct = (i / total_races) * 100
    print(f"[{i+1}/{total_races}] ({pct:.0f}%) {season} R{round_num:02d} {event_name} | {eta_str}", flush=True)

    # Load session
    try:
        session = fastf1.get_session(season, round_num, "R")
        session.load(telemetry=True, weather=False, messages=False)
    except Exception as e:
        errors.append(f"{race_id}: {e}")
        print(f"  ❌ Failed to load: {e}")
        continue

    # Driver mapping from session results
    driver_map = session_driver_mapping(session)

    # Telemetry
    tel_rows = extract_telemetry(session, race_id, driver_map)
    if tel_rows:
        all_telemetry.extend(tel_rows)
        print(f"  ✅ {len(tel_rows):,} telemetry rows")
    else:
        print(f"  ⚠️ No telemetry")

    # Track status
    status_rows = extract_track_status(session, race_id)
    if status_rows:
        all_track_status.extend(status_rows)

    # Circuit geometry
    if race_id not in all_circuits:
        circuit_rows = extract_circuit_geometry(session, race_id, event_name)
        if circuit_rows:
            all_circuits[race_id] = circuit_rows

    # Periodic save (every 20 races) as backup
    if (i + 1) % 20 == 0 and all_telemetry:
        _df = pd.DataFrame(all_telemetry)
        _df.to_parquet(OUTPUT_DIR / "telemetry_checkpoint.parquet", index=False)
        print(f"  💾 Checkpoint saved: {len(all_telemetry):,} rows total")

elapsed_total = time.time() - start_time
print(f"\n{'='*60}")
print(f"DONE in {int(elapsed_total//60)}m {int(elapsed_total%60)}s")
print(f"Telemetry rows: {len(all_telemetry):,}")
print(f"Circuits: {len(all_circuits)}")
print(f"Errors: {len(errors)}")
if errors:
    for e in errors:
        print(f"  - {e}")

In [ ]:
# ===== SAVE FINAL PARQUET FILES =====

# --- telemetry.parquet ---
if all_telemetry:
    tel_df = pd.DataFrame(all_telemetry)
    tel_df["t"] = tel_df["t_raw"] - tel_df.groupby("race_id")["t_raw"].transform("min")
    tel_df = tel_df.drop(columns=["t_raw"])
    tel_df["throttle"] = tel_df["throttle"].clip(0, 100)
    tel_df = tel_df.astype({
        "lap_number": "int16", "sample_index": "int8",
        "x": "float32", "y": "float32", "distance": "float32",
        "speed": "float32", "throttle": "uint8", "brake": "uint8", "t": "float32",
    })
    tel_df = tel_df.sort_values(["race_id", "driver", "lap_number", "sample_index"]).reset_index(drop=True)
    tel_df.to_parquet(OUTPUT_DIR / "telemetry.parquet", index=False, row_group_size=200_000)
    size_mb = (OUTPUT_DIR / "telemetry.parquet").stat().st_size / 1024 / 1024
    print(f"✅ telemetry.parquet: {len(tel_df):,} rows ({size_mb:.1f} MB)")
    print(f"   Races: {tel_df['race_id'].nunique()}")
    print(f"   Drivers: {tel_df['driver'].nunique()}")
else:
    print("❌ No telemetry data!")

# --- track_status.parquet ---
if all_track_status:
    ts_df = pd.DataFrame(all_track_status)
    ts_df.to_parquet(OUTPUT_DIR / "track_status.parquet", index=False)
    print(f"✅ track_status.parquet: {len(ts_df):,} rows")

# --- circuits.parquet ---
circuit_rows_all = [r for rows in all_circuits.values() for r in rows]
if circuit_rows_all:
    circ_df = pd.DataFrame(circuit_rows_all)
    circ_df = circ_df.astype({
        "rotation": "float32", "point_index": "int16",
        "x": "float32", "y": "float32", "distance": "float32",
    })
    circ_df.to_parquet(OUTPUT_DIR / "circuits.parquet", index=False)
    print(f"✅ circuits.parquet: {len(circ_df):,} rows ({len(all_circuits)} circuits)")

print(f"\n📁 Files saved to {OUTPUT_DIR}")
!ls -lh /content/output/*.parquet

In [ ]:
# ===== DOWNLOAD FILES =====
# Run this cell to download the parquet files to your computer.
# If auto-download doesn't work, go to the file browser (folder icon on left)
# and navigate to /content/output/ to download manually.

from google.colab import files

for fname in ["telemetry.parquet", "circuits.parquet", "track_status.parquet"]:
    fpath = OUTPUT_DIR / fname
    if fpath.exists():
        print(f"Downloading {fname}...")
        files.download(str(fpath))
    else:
        print(f"⚠️ {fname} not found")